In [ ]:
import rasterio
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
HEIGHT = 2404
WIDTH = 8344

PATCH_HEIGHT = 601
PATCH_WIDTH = 149

TRAINING_X_MIN = 1192
TRAINING_X_MAX = 5959
TRAINING_Y_MIN = 1202
TRAINING_Y_MAX = 2404

In [ ]:
#rgb data was stored in parts, the data is opened, concatenated correctly and upsampled
path  = ...
parts = []
for i in range(1,14,2):
    with rasterio.open(f"{path}{i}.tif") as x:
        upper = x.read(out_shape=(3,1202,1192))
    with rasterio.open(f"{path}{i+1}.tif") as x:
        lower = x.read(out_shape=(3,1202,1192))
    parts.append(np.concatenate([upper,lower],1))
#divide by 255 to rescale
img = np.concatenate(parts,2).transpose((1,2,0)) / 255


In [ ]:
#lms data
path  = ...
parts = []
for i in range(1,4):
    with rasterio.open(f"{path}{i}.tif") as x:
        parts.append(x.read())
    
lms = np.concatenate(parts,0).transpose((1,2,0))
#if is_ok = False -> outlier
is_ok  = lms < 1e4
minimum = lms.min()
#clean the outliers
lms = lms*is_ok + ~is_ok*minimum
for i,band in enumerate(lms):
    minimum = band.min()
    maximum = band.max()
    #rescale
    band = (band - minimum)/(maximum-minimum)
    lms[i] = band

In [ ]:
path  = ...

with rasterio.open(path) as x:
    hsi = x.read(out_shape=(50,2404,8344))
#correct bands are stored in 0:48
hsi = hsi[0:48]
hsi = hsi.astype(np.float64).transpose((1,2,0))


for i,band in enumerate(hsi):
    minimum = band.min()
    maximum = band.max()
    #rescale
    band = (band - minimum)/(maximum-minimum)
    hsi[i] = band

In [ ]:
#testing mask has no labels in the training area, seperate data needed
path  = ...

with rasterio.open(path) as x:
    mask = x.read()


In [ ]:
#training GT is in seperate file
path  = ...

with rasterio.open(path) as x:
    train_mask = x.read()


In [ ]:
curr_id = 0
path = ...

test_or_train = 'test\\'
#save patches in seperate folders for testing and training data
for i in range(0,HEIGHT,PATCH_HEIGHT):
    for j in range(0,WIDTH,PATCH_WIDTH):
        if TRAINING_Y_MIN <= i <= TRAINING_Y_MAX and TRAINING_X_MIN <= j <= TRAINING_X_MAX:
            test_or_train = 'train\\'
            x = j -TRAINING_X_MIN
            y = i - TRAINING_Y_MIN
            np.save(path + "masks\\" + test_or_train + "mask" + str(curr_id) + ".npy",train_mask[y:y+PATCH_HEIGHT,x:x+PATCH_WIDTH])
        else:
            test_or_train = 'test\\'
            np.save(path + "masks\\" + test_or_train + "mask" + str(curr_id) + ".npy",mask[i:i+PATCH_HEIGHT,j:j+PATCH_WIDTH])
        np.save(path + "hsi\\" + test_or_train + "hsi" + str(curr_id) + ".npy",hsi[i:i+PATCH_HEIGHT,j:j+PATCH_WIDTH])
        np.save(path + "lms\\" + test_or_train + "lms" + str(curr_id) + ".npy",lms[i:i+PATCH_HEIGHT,j:j+PATCH_WIDTH])
        np.save(path + "rgb\\" + test_or_train + "rgb" + str(curr_id) + ".npy",img[i:i+PATCH_HEIGHT,j:j+PATCH_WIDTH])
        
        curr_id += 1